# Week 6 - Phase 2 coverage: ABC classes, movers, branch throughput and forecast readiness
**Questions.** The programme's Phase 2 lists ABC classification, fast and slow movers and warehouse throughput, and Phase 3 will need forecasts. Weeks 4 and 5 answered margin, concentration, lead-time and ledger questions, not these. The team confirmed all three coverage items for Week 6, using sales and branch data only. Week 6 also asks which monthly series are ready to forecast.

**Scope (confirmed by the team).** Sales lines and branch data only. No stock, turnover or cover claims: those depend on the open stock-scale assumption and are handled in the stock work.

**Definitions.**
- **Sales basis:** delivered orders only (18,033 of 20,000 orders; the 1,967 cancelled orders are set aside, as in Weeks 3 and 5). Revenue is `line_total` (before GST); gross profit is `line_profit`.
- **An order** is one `so_id`; every order has one status, one branch and one date.
- **ABC convention:** rank by the measure, add up the shares, and call an item **A** if the share accumulated before it is under 80%, **B** if under 95%, otherwise **C**. The item that crosses a cut-off belongs to the class it reaches.
- **Month:** the calendar month of `order_date`; the data covers 72 months, January 2019 to December 2024.

**Method.** Counts and shares for the classes; Cohen's kappa for agreement; bootstrap intervals (5,000 resamples of months) for monthly volumes; a permutation test (5,000) for whether product volumes differ more than chance; Kruskal-Wallis with an effect size for branch differences; Spearman trend tests with Holm adjustment; a rolling-origin back-test for forecast readiness. Seed 42.

In [1]:
import pandas as pd, numpy as np, os
from pathlib import Path
from scipy import stats
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
SURF,INK,INK2,GRID,BLUE,ORANGE,AQUA,MUTED="#fcfcfb","#0b0b0b","#52514e","#e4e3df","#2a78d6","#eb6834","#1baf7a","#b9b8b2"
def find_dir(env_var, marker, rels):
    # Environment variable first, then this folder and every parent folder, using the repository layout.
    if os.environ.get(env_var) and (Path(os.environ[env_var]) / marker).exists():
        return Path(os.environ[env_var])
    for base in [Path.cwd(), *Path.cwd().parents]:
        for rel in rels:
            if (base / rel / marker).exists():
                return base / rel
    return None
DATA_DIR = find_dir("CADETX_MERGED", "sales_orders_full.csv", ["merged", "data/merged", "week-02/reports/week_02/merged", "reports/week_02/merged"])
assert DATA_DIR is not None, "Could not find the merged datasets. Run from inside the repository, put a merged/ folder next to the notebook, or set the CADETX_MERGED environment variable to the folder."
OUT=Path("coverage_outputs"); OUT.mkdir(exist_ok=True); CH=Path("coverage_charts"); CH.mkdir(exist_ok=True)
SEED=42; NPERM=5000; NBOOT=5000
rng=np.random.default_rng(SEED)
checks=[]
def check(name,expected,actual,ok):
    checks.append(dict(Check_Item=name,Expected=expected,Actual=actual,Status="PASS" if ok else "FAIL")); assert ok,f"{name}: expected {expected}, got {actual}"
def holm(p):
    p=np.asarray(p,float); m=len(p); o=np.argsort(p); adj=np.empty(m); run=0
    for r,i in enumerate(o): run=max(run,(m-r)*p[i]); adj[i]=min(1,run)
    return adj
def kappa(a,b):
    a=np.asarray(a); b=np.asarray(b); labs=sorted(set(a)|set(b)); po=(a==b).mean()
    pe=sum((a==l).mean()*(b==l).mean() for l in labs); return float((po-pe)/(1-pe)) if pe<1 else 1.0
def gini(x):
    x=np.sort(np.asarray(x,float)); n=len(x); return float((2*np.arange(1,n+1)-n-1).dot(x)/(n*x.sum()))
def kw_eps(groups):
    H,p=stats.kruskal(*groups); n=sum(len(g) for g in groups); k=len(groups); return float(H),float(p),float(max(0.0,(H-k+1)/(n-k)))
def style(ax):
    ax.set_facecolor(SURF); ax.figure.set_facecolor(SURF)
    for s_ in ("top","right"): ax.spines[s_].set_visible(False)
    for s_ in ("left","bottom"): ax.spines[s_].set_color(GRID)
    ax.tick_params(colors=INK2); ax.yaxis.grid(True,color=GRID,lw=0.8); ax.set_axisbelow(True)
print("Merged datasets found in:",Path(*DATA_DIR.resolve().parts[-4:]))

Merged datasets found in: week-02/reports/week_02/merged


## Part 0. Load and integrity checks

In [2]:
s=pd.read_csv(DATA_DIR/"sales_orders_full.csv",low_memory=False,parse_dates=["order_date"])
check("sales lines",130402,len(s),len(s)==130402)
check("orders",20000,s.so_id.nunique(),s.so_id.nunique()==20000)
check("products",30,s.product_id.nunique(),s.product_id.nunique()==30)
check("branches",6,s.branch_id.nunique(),s.branch_id.nunique()==6)
mixed=int((s.groupby("so_id").order_status.nunique()>1).sum())
check("every order has one status",0,mixed,mixed==0)
for col in ["branch_id","order_date"]:
    n=int((s.groupby("so_id")[col].nunique()>1).sum()); check(f"every order has one {col}",0,n,n==0)
dl=s[s.order_status=="Delivered"].copy()
check("delivered orders",18033,dl.so_id.nunique(),dl.so_id.nunique()==18033)
check("cancelled orders set aside",1967,s[s.order_status=="Cancelled"].so_id.nunique(),s[s.order_status=="Cancelled"].so_id.nunique()==1967)
months=pd.period_range("2019-01","2024-12",freq="M")
dl["ym"]=dl.order_date.dt.to_period("M")
check("months covered",72,dl.ym.nunique(),dl.ym.nunique()==72 and set(dl.ym.unique())==set(months))
# the delivered orders are the invoiced orders: the totals must reconcile to the invoice file, and to the Q4 base
inv=pd.read_csv(DATA_DIR/"invoices_with_payments.csv",low_memory=False,usecols=["invoice_id","grand_total","payment_total_reliable"])
gt=dl.drop_duplicates("so_id").grand_total.sum(); ig=inv.grand_total.sum()
check("delivered orders = invoices (count)",len(inv),dl.so_id.nunique(),len(inv)==dl.so_id.nunique())
check("delivered order value incl GST equals the invoice file (Rs B)",round(ig/1e9,2),round(gt/1e9,2),abs(gt-ig)<1)
rel_val=inv[inv.payment_total_reliable.astype(str).str.lower().isin(["true","1"])].grand_total.sum()/1e9
check("reliable-invoice value matches the Q4 base (Rs B)",28.60,round(rel_val,2),abs(rel_val-28.60)<0.005)
check("line_total = quantity x unit_price on every line",0,int((dl.quantity*dl.unit_price!=dl.line_total).sum()),int((dl.quantity*dl.unit_price!=dl.line_total).sum())==0)
print(f"Delivered: {dl.so_id.nunique():,} orders, {len(dl):,} lines, revenue Rs {dl.line_total.sum()/1e9:.2f}B before GST, gross profit Rs {dl.line_profit.sum()/1e9:.2f}B")

Delivered: 18,033 orders, 117,656 lines, revenue Rs 22.96B before GST, gross profit Rs 4.95B


## Part 1. ABC classes for the 30 products
Two views, as agreed: by revenue and by gross profit. The class is set by the cumulative share, so the A class is the smallest set of products that makes up 80% of the measure.

In [3]:
P=dl.groupby("product_id").agg(product_name=("product_name","first"),category=("category","first"),unit_price=("unit_price","median"),margin_pct=("margin_percentage","first"),units=("quantity","sum"),orders=("so_id","nunique"),revenue=("line_total","sum"),gross_profit=("line_profit","sum")).reset_index()
def abc(values,a=0.80,b=0.95):
    v=pd.Series(np.asarray(values,float)); order=v.sort_values(ascending=False).index
    cum=v[order].cumsum()/v.sum(); prior=cum.shift(fill_value=0.0)
    cls=pd.Series(np.where(prior<a,"A",np.where(prior<b,"B","C")),index=order)
    return cls.reindex(v.index).values, cum.reindex(v.index).values
P["class_revenue"],P["cum_revenue"]=abc(P.revenue)
P["class_gross_profit"],P["cum_gross_profit"]=abc(P.gross_profit)
P["share_revenue_pct"]=P.revenue/P.revenue.sum()*100; P["share_gross_profit_pct"]=P.gross_profit/P.gross_profit.sum()*100
P=P.sort_values("revenue",ascending=False).reset_index(drop=True); P.insert(0,"rank_revenue",range(1,31))
P.to_csv(OUT/"abc_products.csv",index=False)
# independent recomputation with a plain loop
def abc_loop(vals,ids):
    tot=sum(vals); out={}; run=0.0
    for i,v in sorted(zip(ids,vals),key=lambda t:-t[1]):
        out[i]="A" if run<0.80*tot else ("B" if run<0.95*tot else "C"); run+=v
    return out
for m in ["revenue","gross_profit"]:
    lp=abc_loop(P[m].tolist(),P.product_id.tolist()); same=all(lp[i]==c for i,c in zip(P.product_id,P[f"class_{m}"]))
    check(f"ABC by {m} matches an independent loop",True,same,same)
check("product revenue adds up to delivered revenue",int(dl.line_total.sum()),int(P.revenue.sum()),int(P.revenue.sum())==int(dl.line_total.sum()))
check("product gross profit adds up to delivered gross profit",int(dl.line_profit.sum()),int(P.gross_profit.sum()),int(P.gross_profit.sum())==int(dl.line_profit.sum()))
rows=[]
for m in ["revenue","gross_profit"]:
    for c in "ABC":
        g=P[P[f"class_{m}"]==c]; rows.append(dict(measure=m,abc_class=c,products=len(g),share_of_products_pct=len(g)/30*100,share_of_measure_pct=g[m].sum()/P[m].sum()*100,gross_margin_pct=g.gross_profit.sum()/g.revenue.sum()*100,units_share_pct=g.units.sum()/P.units.sum()*100))
SUM=pd.DataFrame(rows); SUM.to_csv(OUT/"abc_summary.csv",index=False)
for m in ["revenue","gross_profit"]:
    t=SUM[SUM.measure==m]; check(f"classes by {m} cover all 30 products",30,int(t.products.sum()),int(t.products.sum())==30)
    a=t[t.abc_class=="A"].share_of_measure_pct.iloc[0]; check(f"A class by {m} reaches at least 80%",">=80",round(a,1),a>=80)
print(SUM.round(1).to_string(index=False))
print(f"Top 3 products make {P.revenue.head(3).sum()/P.revenue.sum()*100:.1f}% of revenue; top 6 make {P.revenue.head(6).sum()/P.revenue.sum()*100:.1f}%; Gini of product revenue {gini(P.revenue):.2f}")

     measure abc_class  products  share_of_products_pct  share_of_measure_pct  gross_margin_pct  units_share_pct
     revenue         A        10                   33.3                  81.5              20.3             33.5
     revenue         B         7                   23.3                  14.1              25.5             23.1
     revenue         C        13                   43.3                   4.4              31.8             43.4
gross_profit         A        12                   40.0                  82.7              20.6             40.1
gross_profit         B         6                   20.0                  12.6              25.8             19.9
gross_profit         C        12                   40.0                   4.6              33.2             40.0
Top 3 products make 44.3% of revenue; top 6 make 65.1%; Gini of product revenue 0.63


In [4]:
# agreement between the two views, sensitivity to the cut-offs, and stability over time
ct=pd.crosstab(P.class_revenue,P.class_gross_profit); ct.to_csv(OUT/"abc_revenue_vs_gross_profit.csv")
k_view=kappa(P.class_revenue,P.class_gross_profit); same_view=int((P.class_revenue==P.class_gross_profit).sum())
print(f"Revenue class equals gross-profit class for {same_view} of 30 products (kappa {k_view:.2f})")
print(ct.to_string())
sens=[]
for a_,b_ in [(0.70,0.90),(0.80,0.95),(0.85,0.97)]:
    for m in ["revenue","gross_profit"]:
        c_,_=abc(P[m],a_,b_); u=pd.Series(c_).value_counts()
        sens.append(dict(cutoffs=f"{int(a_*100)}/{int(b_*100)}",measure=m,A=int(u.get("A",0)),B=int(u.get("B",0)),C=int(u.get("C",0))))
SENS=pd.DataFrame(sens); SENS.to_csv(OUT/"abc_cutoff_sensitivity.csv",index=False); print(SENS.to_string(index=False))
half={}
for nm,yrs in [("2019-2021",[2019,2020,2021]),("2022-2024",[2022,2023,2024])]:
    h=dl[dl.order_date.dt.year.isin(yrs)].groupby("product_id").line_total.sum().reindex(P.product_id).values
    half[nm]=abc(h)[0]
k_time=kappa(half["2019-2021"],half["2022-2024"]); moved=int((half["2019-2021"]!=half["2022-2024"]).sum())
pd.DataFrame(dict(product_id=P.product_id,class_2019_2021=half["2019-2021"],class_2022_2024=half["2022-2024"])).to_csv(OUT/"abc_stability_by_period.csv",index=False)
print(f"Revenue class in 2019-2021 vs 2022-2024: {30-moved} of 30 products keep their class (kappa {k_time:.2f}); {moved} change")
# what drives the concentration: price or volume?
rho_price=stats.spearmanr(P.revenue,P.unit_price)[0]; rho_units=stats.spearmanr(P.revenue,P.units)[0]
print(f"Revenue rank vs unit price: Spearman {rho_price:+.2f}; vs units sold: {rho_units:+.2f}")
CAT=P.groupby("category").agg(products=("product_id","count"),revenue_share_pct=("share_revenue_pct","sum"),gross_profit_share_pct=("share_gross_profit_pct","sum"),a_class_products=("class_revenue",lambda x:int((x=="A").sum()))).sort_values("revenue_share_pct",ascending=False).reset_index()
CAT.to_csv(OUT/"abc_by_category.csv",index=False); print(CAT.round(1).to_string(index=False))
check("Cohen kappa between the two views is a valid value",True,round(k_view,3),-1<=k_view<=1)

Revenue class equals gross-profit class for 27 of 30 products (kappa 0.85)
class_gross_profit   A  B   C
class_revenue                
A                   10  0   0
B                    2  5   0
C                    0  1  12
cutoffs      measure  A  B  C
  70/90      revenue  7  7 16
  70/90 gross_profit  9  7 14
  80/95      revenue 10  7 13
  80/95 gross_profit 12  6 12
  85/97      revenue 12  6 12
  85/97 gross_profit 13  7 10
Revenue class in 2019-2021 vs 2022-2024: 30 of 30 products keep their class (kappa 1.00); 0 change


Revenue rank vs unit price: Spearman +1.00; vs units sold: +0.17
     category  products  revenue_share_pct  gross_profit_share_pct  a_class_products
       Engine         4               33.0                    30.8                 3
    Hydraulic         6               28.4                    29.8                 5
 Transmission         1               14.8                    12.3                 1
Undercarriage         4               13.1                    13.9                 1
   Electrical         3                5.1                     5.6                 0
      Cooling         1                1.9                     2.0                 0
        Cabin         1                1.4                     1.9                 0
      Sensors         1                0.7                     1.0                 0
       Brakes         1                0.4                     0.5                 0
      Filters         2                0.4                     0.6                 0


## Part 2. Fast, medium and slow movers
Movers are set by **monthly units sold**. Before labelling any product fast or slow, the notebook tests whether products differ in volume by more than chance, because a label means nothing if the ranking is noise. A product is called fast or slow only if its 95% bootstrap interval for mean monthly units lies wholly above or below the average product.

In [5]:
M=dl.groupby(["product_id","ym"]).quantity.sum().unstack(fill_value=0).reindex(index=P.product_id,columns=months,fill_value=0)
check("product-month grid is 30 x 72",(30,72),M.shape,M.shape==(30,72))
check("no product-month has zero sales",0,int((M.values==0).sum()),int((M.values==0).sum())==0)
check("monthly units add up to delivered units",int(dl.quantity.sum()),int(M.values.sum()),int(M.values.sum())==int(dl.quantity.sum()))
mean_u=M.mean(axis=1).values; grand=mean_u.mean()
idx=rng.integers(0,72,(NBOOT,72)); boot=M.values[:,idx].mean(axis=2)
lo,hi=np.percentile(boot,[2.5,97.5],axis=1)
mv=pd.DataFrame(dict(product_id=P.product_id.values,product_name=P.product_name.values,mean_monthly_units=mean_u,ci_low=lo,ci_high=hi,sd_monthly_units=M.std(axis=1).values,margin_pct=P.margin_pct.values,revenue=P.revenue.values))
mv["vs_average_pct"]=(mv.mean_monthly_units/grand-1)*100
mv["interval_rule_class"]=np.where(mv.ci_low>grand,"Fast",np.where(mv.ci_high<grand,"Slow","Medium (not different from average)"))
r=mv.mean_monthly_units.rank(ascending=False,method="first"); mv["rank_tertile_reference"]=np.where(r<=10,"top third",np.where(r<=20,"middle third","bottom third"))
mv=mv.sort_values("mean_monthly_units",ascending=False).reset_index(drop=True); mv.to_csv(OUT/"movers_products.csv",index=False)
H,p_kw,eps=kw_eps([M.values[i] for i in range(30)])
# permutation: shuffle product labels across delivered lines; does the spread of product totals exceed chance?
lab=pd.factorize(dl.product_id)[0]; q=dl.quantity.values.astype(float)
obs_sd=np.bincount(lab,weights=q).std()
perm_sd=np.array([np.bincount(rng.permutation(lab),weights=q).std() for _ in range(NPERM)])
p_perm=float((1+(perm_sd>=obs_sd).sum())/(NPERM+1))
spread=(mean_u.max()/mean_u.min()-1)*100; cv_mean=mean_u.std()/grand
half_u={}
for nm,yrs in [("a",[2019,2020,2021]),("b",[2022,2023,2024])]:
    half_u[nm]=dl[dl.order_date.dt.year.isin(yrs)].groupby("product_id").quantity.sum().reindex(P.product_id).values
rho_stab,p_stab=stats.spearmanr(half_u["a"],half_u["b"])
rho_m,p_m=stats.spearmanr(P.units,P.margin_pct); rho_rm,p_rm=stats.spearmanr(P.revenue,P.margin_pct)
volumes_differ=bool(p_kw<0.05 and p_perm<0.05)
mv["mover_class"]=mv.interval_rule_class if volumes_differ else "Not distinguishable (volumes equal within chance)"
mv.to_csv(OUT/"movers_products.csv",index=False)
cnt=mv.mover_class.value_counts().to_dict()
nrule=mv.interval_rule_class.value_counts().to_dict()
pd.DataFrame([dict(test="Kruskal-Wallis on monthly units across the 30 products",statistic=H,p_value=p_kw,effect_size=eps,effect_label="epsilon-squared"),
 dict(test="Permutation: SD of product total units vs random product labels",statistic=obs_sd,p_value=p_perm,effect_size=obs_sd/perm_sd.mean(),effect_label="observed SD / mean SD under chance"),
 dict(test="Spearman: product units 2019-2021 vs 2022-2024",statistic=rho_stab,p_value=p_stab,effect_size=rho_stab,effect_label="rho"),
 dict(test="Spearman: product units vs Q6 margin %",statistic=rho_m,p_value=p_m,effect_size=rho_m,effect_label="rho"),
 dict(test="Spearman: product revenue vs Q6 margin %",statistic=rho_rm,p_value=p_rm,effect_size=rho_rm,effect_label="rho")]).to_csv(OUT/"movers_tests.csv",index=False)
check("mover classes cover all 30 products",30,int(sum(cnt.values())),sum(cnt.values())==30)
check("no mover label is given when volumes do not differ beyond chance",True,volumes_differ or all(k.startswith("Not distinguishable") for k in cnt),volumes_differ or all(k.startswith("Not distinguishable") for k in cnt))
check("permutation test uses 5,000 shuffles",NPERM,len(perm_sd),len(perm_sd)==NPERM)
print(f"Mean monthly units per product: {mean_u.min():.0f} to {mean_u.max():.0f} (the busiest product sells {spread:.1f}% more than the quietest); average {grand:.0f}")
print(f"Kruskal-Wallis across products: p = {p_kw:.3f}, epsilon-squared {eps:.3f}")
print(f"Spread of product totals vs random labels: p = {p_perm:.3f} (observed SD is {obs_sd/perm_sd.mean():.2f} times the chance SD)")
print(f"Product ranking 2019-2021 vs 2022-2024: Spearman {rho_stab:+.2f} (p = {p_stab:.2f})")
print(f"Interval rule alone would label: {nrule}; with 30 products about 0.75 Fast and 0.75 Slow are expected by chance. Volumes differ beyond chance: {volumes_differ}")
print(f"Movers: {cnt}")
print(f"Units vs margin %: Spearman {rho_m:+.2f} (p = {p_m:.2f}); revenue vs margin %: {rho_rm:+.2f} (p = {p_rm:.3f})")

Mean monthly units per product: 548 to 593 (the busiest product sells 8.1% more than the quietest); average 572
Kruskal-Wallis across products: p = 0.621, epsilon-squared 0.000
Spread of product totals vs random labels: p = 0.449 (observed SD is 1.01 times the chance SD)
Product ranking 2019-2021 vs 2022-2024: Spearman -0.09 (p = 0.64)
Interval rule alone would label: {'Medium (not different from average)': 28, 'Fast': 1, 'Slow': 1}; with 30 products about 0.75 Fast and 0.75 Slow are expected by chance. Volumes differ beyond chance: False
Movers: {'Not distinguishable (volumes equal within chance)': 30}
Units vs margin %: Spearman -0.24 (p = 0.21); revenue vs margin %: -0.91 (p = 0.000)


## Part 3. Branch throughput: orders and units per branch and month
A throughput view of **delivered** orders and units per branch-month, with a note on what the data cannot show.

In [6]:
BR=s.groupby("branch_id").branch_name.first().to_dict()
B=dl.groupby(["branch_id","ym"]).agg(orders=("so_id","nunique"),units=("quantity","sum"),revenue=("line_total","sum")).reset_index()
grid=pd.MultiIndex.from_product([sorted(BR),months],names=["branch_id","ym"])
B=B.set_index(["branch_id","ym"]).reindex(grid,fill_value=0).reset_index(); B["branch"]=B.branch_id.map(BR)
check("branch-month grid is 6 x 72",432,len(B),len(B)==432)
check("branch orders add up to delivered orders",18033,int(B.orders.sum()),int(B.orders.sum())==18033)
check("branch units add up to delivered units",int(dl.quantity.sum()),int(B.units.sum()),int(B.units.sum())==int(dl.quantity.sum()))
check("no branch-month is empty",0,int((B.orders==0).sum()),int((B.orders==0).sum())==0)
B.assign(month=B.ym.astype(str)).drop(columns="ym").to_csv(OUT/"branch_month_throughput.csv",index=False)
def bci(v):
    v=np.asarray(v,float); b=v[rng.integers(0,len(v),(NBOOT,len(v)))].mean(axis=1); return np.percentile(b,[2.5,97.5])
oq=dl.groupby("so_id").agg(branch_id=("branch_id","first"),units=("quantity","sum"))
rows=[]
for b_ in sorted(BR):
    g=B[B.branch_id==b_]; lo_,hi_=bci(g.orders); ulo,uhi=bci(g.units)
    rows.append(dict(branch_id=b_,branch=BR[b_],orders_total=int(g.orders.sum()),share_of_orders_pct=g.orders.sum()/B.orders.sum()*100,orders_per_month=g.orders.mean(),orders_ci_low=lo_,orders_ci_high=hi_,orders_sd=g.orders.std(),orders_cv=g.orders.std()/g.orders.mean(),min_month=int(g.orders.min()),max_month=int(g.orders.max()),units_per_month=g.units.mean(),units_ci_low=ulo,units_ci_high=uhi,units_per_order=g.units.sum()/g.orders.sum()))
BT=pd.DataFrame(rows).sort_values("orders_per_month",ascending=False).reset_index(drop=True); BT.to_csv(OUT/"branch_throughput_summary.csv",index=False)
Ho,po,eo=kw_eps([B[B.branch_id==b_].orders.values for b_ in sorted(BR)]); Hu,pu,eu=kw_eps([B[B.branch_id==b_].units.values for b_ in sorted(BR)])
Hb,pb,eb=kw_eps([oq[oq.branch_id==b_].units.values for b_ in sorted(BR)])
# trend inside each branch (month index vs orders), Holm over the 6 branches
tr=[]
for b_ in sorted(BR):
    g=B[B.branch_id==b_]; rho,p=stats.spearmanr(np.arange(72),g.orders); tr.append(dict(branch_id=b_,branch=BR[b_],spearman_rho=rho,p_value=p))
TR=pd.DataFrame(tr); TR["holm_p"]=holm(TR.p_value); TR.to_csv(OUT/"branch_trend_tests.csv",index=False)
Y=B.assign(year=B.ym.dt.year).groupby(["branch","year"]).orders.mean().unstack().round(1); Y.to_csv(OUT/"branch_orders_per_month_by_year.csv")
# sensitivity: include cancelled orders
allB=s.assign(ym=s.order_date.dt.to_period("M")).groupby("branch_id").so_id.nunique()
rho_s=stats.spearmanr(BT.set_index("branch_id").orders_total.reindex(sorted(BR)),allB.reindex(sorted(BR)))[0]
pd.DataFrame([dict(test="Kruskal-Wallis, orders per branch-month",statistic=Ho,p_value=po,effect_size=eo,effect_label="epsilon-squared"),
 dict(test="Kruskal-Wallis, units per branch-month",statistic=Hu,p_value=pu,effect_size=eu,effect_label="epsilon-squared"),
 dict(test="Kruskal-Wallis, units per order across branches",statistic=Hb,p_value=pb,effect_size=eb,effect_label="epsilon-squared")]).to_csv(OUT/"branch_tests.csv",index=False)
check("branch table has 6 rows",6,len(BT),len(BT)==6)
check("branch shares add up to 100%",100.0,round(BT.share_of_orders_pct.sum(),6),abs(BT.share_of_orders_pct.sum()-100)<1e-6)
print(BT[["branch","orders_per_month","orders_ci_low","orders_ci_high","units_per_month","units_per_order"]].round(1).to_string(index=False))
print(f"Branch differences, orders per month: p = {po:.2g}, epsilon-squared {eo:.2f}; units per month: epsilon-squared {eu:.2f}; units per order: p = {pb:.2f}, epsilon-squared {eb:.3f}")
print("Trend inside each branch (Spearman, Holm):",TR.assign(rho=TR.spearman_rho.round(2),holm=TR.holm_p.round(2))[["branch","rho","holm"]].to_dict("records"))
print(f"Busiest vs quietest branch: {BT.orders_per_month.iloc[0]/BT.orders_per_month.iloc[-1]:.2f} times the monthly orders; ranking with cancelled orders included keeps the same order (Spearman {rho_s:+.2f})")

             branch  orders_per_month  orders_ci_low  orders_ci_high  units_per_month  units_per_order
  Chennai South Hub              51.1           49.1            53.1           3526.7             69.1
 Kolkata East Depot              47.4           45.7            49.1           3206.5             67.6
 Ahmedabad West Hub              43.3           41.7            44.9           2975.3             68.8
      Delhi Central              37.2           36.0            38.6           2573.9             69.1
  Pune Distribution              37.1           35.7            38.5           2522.7             68.1
Hyderabad Logistics              34.4           33.1            35.7           2346.9             68.2
Branch differences, orders per month: p = 2e-39, epsilon-squared 0.44; units per month: epsilon-squared 0.38; units per order: p = 0.57, epsilon-squared 0.000


Trend inside each branch (Spearman, Holm): [{'branch': 'Ahmedabad West Hub', 'rho': 0.03, 'holm': 1.0}, {'branch': 'Chennai South Hub', 'rho': -0.13, 'holm': 1.0}, {'branch': 'Delhi Central', 'rho': 0.1, 'holm': 1.0}, {'branch': 'Hyderabad Logistics', 'rho': -0.02, 'holm': 1.0}, {'branch': 'Kolkata East Depot', 'rho': -0.04, 'holm': 1.0}, {'branch': 'Pune Distribution', 'rho': -0.03, 'holm': 1.0}]
Busiest vs quietest branch: 1.48 times the monthly orders; ranking with cancelled orders included keeps the same order (Spearman +0.94)


## Part 4. Which series are ready to forecast?
Phase 3 will forecast demand. This part asks, for each monthly series, whether anything in the history (a trend, a seasonal pattern, a break) would justify more than a simple level forecast, and tests it with a **rolling-origin back-test**: from month 49 onward, each month is forecast one step ahead using only earlier months, by five simple methods. A method counts as an improvement only if it beats the plain average of earlier months and the 95% bootstrap interval of the paired error difference stays above zero.

Calendar effect: months have different numbers of days, which alone makes February look low. Seasonality is therefore tested on **per-day rates** as well as on raw totals.

In [7]:
def acf(x,k):
    x=np.asarray(x,float)-np.mean(x); return float((x[:-k]*x[k:]).sum()/(x*x).sum())
METHODS={"mean_of_all_earlier":lambda h:h.mean(),"trailing_12_month_mean":lambda h:h[-12:].mean(),"last_value":lambda h:h[-1],"seasonal_naive":lambda h:h[-12],"linear_trend":lambda h:float(np.polyval(np.polyfit(np.arange(len(h)),h,1),len(h)))}
START=48
def backtest(v):
    v=np.asarray(v,float); return {k:np.array([abs(v[i]-f(v[:i])) for i in range(START,len(v))]) for k,f in METHODS.items()}
def beats(err,m):
    d=err["mean_of_all_earlier"]-err[m]; b=d[rng.integers(0,len(d),(NBOOT,len(d)))].mean(axis=1); lo_,hi_=np.percentile(b,[2.5,97.5]); return float(d.mean()),float(lo_),float(hi_),bool(lo_>0)
dim=months.days_in_month.values
def month_tests(v):
    v=np.asarray(v,float); perday=v/dim; mo=months.month.values
    Hr,pr,er=kw_eps([v[mo==m] for m in range(1,13)]); Hd,pd_,ed=kw_eps([perday[mo==m] for m in range(1,13)])
    rho,pt=stats.spearmanr(np.arange(72),v); return dict(cv=float(v.std()/v.mean()),trend_rho=float(rho),trend_p=float(pt),season_raw_p=float(pr),season_perday_p=float(pd_),season_perday_eps=float(ed))
TOT={"revenue":dl.groupby("ym").line_total.sum().reindex(months).values,"units":dl.groupby("ym").quantity.sum().reindex(months).values,"orders":dl.groupby("ym").so_id.nunique().reindex(months).values,"gross_profit":dl.groupby("ym").line_profit.sum().reindex(months).values}
for k,v in TOT.items(): check(f"{k} series has 72 months",72,len(v),len(v)==72 and not np.isnan(v.astype(float)).any())
series={("all",k):v for k,v in TOT.items()}
for i,pid in enumerate(P.product_id): series[("product",pid)]=M.loc[pid].values
for b_ in sorted(BR): series[("branch",b_)]=B[B.branch_id==b_].orders.values
rows=[]; bt_tot=[]
for (grp,name),v in series.items():
    t=month_tests(v); e=backtest(v); base=np.mean(e["mean_of_all_earlier"])
    row=dict(group=grp,series=name,**t)
    for m in METHODS: row[f"mae_pct_{m}"]=float(e[m].mean()/np.mean(v)*100)
    imp=[]
    for m in list(METHODS)[1:]:
        dm,lo_,hi_,ok=beats(e,m); row[f"beats_mean_{m}"]=ok
        if ok: imp.append(m)
    row["improving_methods"]=", ".join(imp) if imp else "none"
    row["forecast_interval_halfwidth_pct"]=float(1.96*np.std(v,ddof=1)/np.mean(v)*100)
    rows.append(row)
    if grp=="all":
        for m in METHODS: bt_tot.append(dict(series=name,method=m,mae_pct_of_mean=float(e[m].mean()/np.mean(v)*100)))
R=pd.DataFrame(rows)
for g_ in ["product","branch"]:
    m_=R.group==g_; R.loc[m_,"trend_holm_p"]=holm(R.loc[m_,"trend_p"]); R.loc[m_,"season_perday_holm_p"]=holm(R.loc[m_,"season_perday_p"])
R["trend_flag"]=np.where(R.group=="all",R.trend_p<0.05,R.trend_holm_p<0.05); R["season_flag"]=np.where(R.group=="all",R.season_perday_p<0.05,R.season_perday_holm_p<0.05)
R["readiness"]=np.where(R.trend_flag|R.season_flag,"Review: trend or seasonality found","Ready: level forecast (average with prediction interval)")
R.to_csv(OUT/"forecast_readiness_series.csv",index=False); pd.DataFrame(bt_tot).to_csv(OUT/"forecast_backtest_totals.csv",index=False)
nforecast=len(range(START,72)); check("back-test makes 24 one-step forecasts per method",24,nforecast,nforecast==24)
check("series assessed (4 totals, 30 products, 6 branches)",40,len(R),len(R)==40)
# the plain-average baseline reproduced by hand for one series
v=TOT["orders"].astype(float); hand=np.mean([abs(v[i]-v[:i].mean()) for i in range(48,72)]); check("baseline error reproduces by hand",round(hand,6),round(float(backtest(v)["mean_of_all_earlier"].mean()),6),abs(hand-backtest(v)["mean_of_all_earlier"].mean())<1e-9)
T=R[R.group=="all"].set_index("series")
print(T[["cv","trend_rho","trend_p","season_raw_p","season_perday_p","season_perday_eps"]].round(3).to_string())
print(T[[c for c in T.columns if c.startswith("mae_pct_")]].round(1).to_string())
print("Totals where any method beats the plain average:",T.improving_methods.to_dict())
print("Readiness:",R.groupby("group").readiness.value_counts().to_dict())
print("95% prediction half-width as % of the monthly average (about +/-):",R.groupby("group").forecast_interval_halfwidth_pct.median().round(0).to_dict())
for g_ in ["product","branch"]:
    G=R[R.group==g_]; ncmp=len(G)*4; nb=int(G[[c for c in G.columns if c.startswith("beats_mean_")]].values.sum())
    print(f"{g_}: {len(G)} series, median CV {G.cv.median():.2f}; trend significant after Holm: {int((G.trend_holm_p<0.05).sum())}; per-day seasonality significant after Holm: {int((G.season_perday_holm_p<0.05).sum())}; method beats the plain average in {nb} of {ncmp} comparisons (about {ncmp*0.025:.1f} expected by chance)")

                 cv  trend_rho  trend_p  season_raw_p  season_perday_p  season_perday_eps
series                                                                                   
revenue       0.082     -0.191    0.108         0.038            0.694              0.000
units         0.080     -0.158    0.185         0.021            0.448              0.000
orders        0.074     -0.035    0.772         0.021            0.415              0.006
gross_profit  0.081     -0.187    0.117         0.042            0.746              0.000
              mae_pct_mean_of_all_earlier  mae_pct_trailing_12_month_mean  mae_pct_last_value  mae_pct_seasonal_naive  mae_pct_linear_trend
series                                                                                                                                     
revenue                               6.7                             6.4                 6.7                     8.2                   6.4
units                                 5.

## Charts

In [8]:
cls_col={"A":BLUE,"B":AQUA,"C":MUTED}
fig,axs=plt.subplots(1,2,figsize=(13,5.2))
for ax,(m,ttl) in zip(axs,[("revenue","By revenue"),("gross_profit","By gross profit")]):
    Q=P.sort_values(m,ascending=False).reset_index(drop=True); cum=Q[m].cumsum()/Q[m].sum()*100
    ax.bar(range(30),Q[m]/Q[m].sum()*100,color=[cls_col[c] for c in Q[f"class_{m}"]],width=0.72)
    ax2=ax.twinx(); ax2.plot(range(30),cum,color=INK,lw=1.8); ax2.set_ylim(0,102); ax2.set_ylabel("Cumulative share (%)",color=INK2)
    for y in (80,95):
        ax2.axhline(y,color=INK2,lw=0.8,ls=":"); ax2.text(29.6,y-3.2,f"{y}%",ha="right",fontsize=8.5,color=INK2)
    ax.set_xticks(range(30)); ax.set_xticklabels(Q.product_id,rotation=90,fontsize=7.5); ax.set_ylabel("Share of total (%)",color=INK2); ax.set_title(ttl,color=INK,loc="left",fontsize=12)
    style(ax); ax2.spines["top"].set_visible(False); ax2.tick_params(colors=INK2)
    na=int((Q[f"class_{m}"]=="A").sum()); nb_=int((Q[f"class_{m}"]=="B").sum())
    ax.text(0.97,0.60,f"A: {na} products\nB: {nb_} products\nC: {30-na-nb_} products",transform=ax.transAxes,ha="right",va="top",fontsize=10,color=INK,bbox=dict(boxstyle="round,pad=0.4",fc=SURF,ec=GRID))
fig.suptitle("ABC classes: a few high-price products make most of the money",color=INK,x=0.01,ha="left",fontsize=13,fontweight="bold")
fig.tight_layout(rect=(0,0,1,0.94)); fig.savefig(CH/"cov_01_abc_pareto.png",dpi=160); plt.close(fig)

fig,ax=plt.subplots(figsize=(11,5.4)); Q=mv.sort_values("mean_monthly_units",ascending=False).reset_index(drop=True)
ax.errorbar(range(30),Q.mean_monthly_units,yerr=[Q.mean_monthly_units-Q.ci_low,Q.ci_high-Q.mean_monthly_units],fmt="o",color=BLUE,ecolor=BLUE,elinewidth=1.4,capsize=2.5,ms=5)
ax.axhline(grand,color=ORANGE,lw=1.6)
fig.text(0.01,0.012,f"Orange line: the average product ({grand:.0f} units). The one product just above it and the one just below are what chance gives for 30 products (permutation test p = {p_perm:.2f}).",fontsize=9.5,color=INK2,ha="left")
ax.set_xticks(range(30)); ax.set_xticklabels(Q.product_id,rotation=90,fontsize=8); ax.set_ylabel("Mean units sold per month (95% interval)",color=INK2)
ax.set_title(f"Movers: every product sells {mean_u.min():.0f} to {mean_u.max():.0f} units a month, so no product is clearly fast or slow",color=INK,loc="left",fontsize=12)
style(ax); fig.tight_layout(rect=(0,0.04,1,1)); fig.savefig(CH/"cov_02_movers.png",dpi=160); plt.close(fig)

fig,axs=plt.subplots(1,2,figsize=(13,5.2)); order_=BT.branch_id.tolist(); lab_=[BR[b_].split()[0] for b_ in order_]
for ax,col,ttl in [(axs[0],"orders","Orders per branch-month"),(axs[1],"units","Units per branch-month")]:
    data=[B[B.branch_id==b_][col].values for b_ in order_]
    bp=ax.boxplot(data,patch_artist=True,widths=0.55,showfliers=False,medianprops=dict(color=INK,lw=1.6)); ax.set_xticks(range(1,len(lab_)+1)); ax.set_xticklabels(lab_)
    for patch in bp["boxes"]: patch.set(facecolor=BLUE,alpha=0.35,edgecolor=BLUE)
    for w_ in bp["whiskers"]+bp["caps"]: w_.set(color=BLUE)
    for i,d_ in enumerate(data): ax.plot(np.full(len(d_),i+1)+rng.uniform(-0.17,0.17,len(d_)),d_,"o",ms=2.5,color=INK2,alpha=0.35)
    ax.set_title(ttl,color=INK,loc="left",fontsize=12); ax.set_ylabel(col.capitalize(),color=INK2); style(ax)
fig.suptitle("Branch throughput: Chennai and Kolkata run busiest, Hyderabad quietest, steady over the six years",color=INK,x=0.01,ha="left",fontsize=13,fontweight="bold")
fig.tight_layout(rect=(0,0,1,0.94)); fig.savefig(CH/"cov_03_branch_throughput.png",dpi=160); plt.close(fig)

fig,ax=plt.subplots(figsize=(10,5.2)); ov=TOT["orders"].astype(float); mo_=months.month.values
raw_idx=np.array([ov[mo_==m].mean() for m in range(1,13)])/ov.mean(); pdv=ov/dim; day_idx=np.array([pdv[mo_==m].mean() for m in range(1,13)])/pdv.mean()
ax.plot(range(1,13),raw_idx,color=ORANGE,lw=2,marker="o",label="Orders per month (raw)"); ax.plot(range(1,13),day_idx,color=BLUE,lw=2,marker="o",label="Orders per day")
ax.axhline(1,color=INK2,lw=0.8,ls=":"); ax.set_xticks(range(1,13)); ax.set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
ax.set_ylabel("Index (average month = 1)",color=INK2); ax.legend(frameon=False,loc="lower right")
ax.set_title("The February dip is month length: per day, no month stands out",color=INK,loc="left",fontsize=12); style(ax)
fig.tight_layout(); fig.savefig(CH/"cov_04_month_of_year.png",dpi=160); plt.close(fig)

fig,ax=plt.subplots(figsize=(11,5.2)); BTT=pd.DataFrame(bt_tot); mnames=list(METHODS); w=0.16; cols=[BLUE,AQUA,MUTED,ORANGE,INK2]
for j,m in enumerate(mnames):
    vals=[BTT[(BTT.series==sr)&(BTT.method==m)].mae_pct_of_mean.iloc[0] for sr in TOT]
    ax.bar(np.arange(4)+(j-2)*w,vals,width=w*0.9,color=cols[j],label=m.replace("_"," "))
ax.set_xticks(range(4)); ax.set_xticklabels([t.replace("_"," ") for t in TOT]); ax.set_ylabel("Average one-step error (% of mean)",color=INK2)
ax.legend(frameon=False,ncol=3,fontsize=9,loc="upper left"); ax.set_ylim(0,max(BTT.mae_pct_of_mean)*1.35)
ax.set_title("Back-test: nothing clearly beats the plain average, and seasonal-naive is the worst",color=INK,loc="left",fontsize=12); style(ax)
fig.tight_layout(); fig.savefig(CH/"cov_05_backtest.png",dpi=160); plt.close(fig)
check("five charts saved",5,len(list(CH.glob("cov_*.png"))),len(list(CH.glob("cov_*.png")))==5)

## Findings

In [9]:
a=SUM[(SUM.measure=="revenue")]; ag=SUM[(SUM.measure=="gross_profit")]
print(f"1. ABC by revenue: A = {int(a[a.abc_class=='A'].products.iloc[0])} products ({a[a.abc_class=='A'].share_of_measure_pct.iloc[0]:.1f}% of revenue), B = {int(a[a.abc_class=='B'].products.iloc[0])}, C = {int(a[a.abc_class=='C'].products.iloc[0])}. By gross profit: A = {int(ag[ag.abc_class=='A'].products.iloc[0])}, B = {int(ag[ag.abc_class=='B'].products.iloc[0])}, C = {int(ag[ag.abc_class=='C'].products.iloc[0])}.")
print(f"   The two views agree for {same_view} of 30 products (kappa {k_view:.2f}); classes hold across periods for {30-moved} of 30 (kappa {k_time:.2f}); concentration follows price (rho {rho_price:+.2f}), not volume (rho {rho_units:+.2f}).")
mg=SUM[SUM.measure=="revenue"].set_index("abc_class")
print(f"   Gross margin by revenue class: A {mg.gross_margin_pct['A']:.1f}%, B {mg.gross_margin_pct['B']:.1f}%, C {mg.gross_margin_pct['C']:.1f}%; A is {mg.share_of_measure_pct['A']:.0f}% of revenue but {mg.units_share_pct['A']:.0f}% of units. Product revenue vs margin %: rho {rho_rm:+.2f}.")
print(f"2. Movers: product volumes differ by {spread:.1f}% from quietest to busiest; Kruskal-Wallis p = {p_kw:.2f}; chance-label test p = {p_perm:.2f}; ranking across periods rho {rho_stab:+.2f}. Classes: {cnt}.")
print(f"3. Branches: {BT.branch.iloc[0]} {BT.orders_per_month.iloc[0]:.1f} orders a month to {BT.branch.iloc[-1]} {BT.orders_per_month.iloc[-1]:.1f}; epsilon-squared {eo:.2f}; basket size does not differ (p = {pb:.2f}).")
print("4. Forecast readiness: see the table above and the note in the report.")
print("LIMITS: sales and branch data only. No stock, capacity, staffing, pick-and-pack or dwell-time data, and delivery dates look generated (Q7), so throughput here means orders and units sold, not warehouse capacity or speed.")

1. ABC by revenue: A = 10 products (81.5% of revenue), B = 7, C = 13. By gross profit: A = 12, B = 6, C = 12.
   The two views agree for 27 of 30 products (kappa 0.85); classes hold across periods for 30 of 30 (kappa 1.00); concentration follows price (rho +1.00), not volume (rho +0.17).
   Gross margin by revenue class: A 20.3%, B 25.5%, C 31.8%; A is 82% of revenue but 34% of units. Product revenue vs margin %: rho -0.91.
2. Movers: product volumes differ by 8.1% from quietest to busiest; Kruskal-Wallis p = 0.62; chance-label test p = 0.45; ranking across periods rho -0.09. Classes: {'Not distinguishable (volumes equal within chance)': 30}.
3. Branches: Chennai South Hub 51.1 orders a month to Hyderabad Logistics 34.4; epsilon-squared 0.44; basket size does not differ (p = 0.57).
4. Forecast readiness: see the table above and the note in the report.
LIMITS: sales and branch data only. No stock, capacity, staffing, pick-and-pack or dwell-time data, and delivery dates look generated (Q

## Validation report

In [10]:
rep=pd.DataFrame(checks); rep.insert(0,"Check_ID",range(1,len(rep)+1)); rep.to_csv("validation_report_week06_coverage.csv",index=False)
print(rep.Status.value_counts().to_dict()); assert (rep.Status=="PASS").all()
print(f"{len(rep)} checks, all PASS")

{'PASS': 43}
43 checks, all PASS
